In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.font_manager as fm
#matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportion_confint
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

#폰트 설정(없으면 영문으로 나옴)
try:
    font_path = [f for f in fm.findSystemFonts() if 'NanumGothic' in f or 'Malgun' in f or 'AppleGothic' in f]
    if font_path:
        plt.rcParams['font.family'] = fm.FontProperties(fname=font_path[0]).get_name()
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

In [6]:
funnel = pd.read_csv('../Soyun_EDA/funnel_instance.csv')#EDA 코드 돌리고 나온 파일로 하심 됩니다
full = pd.read_csv('../Soyun_EDA/final_eda.csv')

In [7]:
portfolio = (
    full[['offer_id', 'ch_mobile', 'ch_web', 'ch_email', 'ch_social', 'channel_count']]
    .dropna(subset=['offer_id'])
    .drop_duplicates('offer_id')
)
funnel = funnel.merge(portfolio, on='offer_id', how='left')

cust_attr = (
    full[['customer_id', 'gender', 'age_group', 'age_gender', 'income_group', 'income']]
    .drop_duplicates('customer_id')
)
funnel = funnel.merge(cust_attr, on='customer_id', how='left', suffixes=('', '_dup'))

funnel = funnel.drop(columns=[
    'gender_dup', 'age_group_dup', 'age_gender_dup', 'income_group_dup', 'income_dup'
], errors='ignore')

# 누락 제외
funnel_clean = funnel[funnel['income_group'] != '누락'].copy()
funnel_clean = funnel_clean[funnel_clean['age_group'] != '누락'].copy()

# amount 이상치 처리: 상위 1% 제거
tx_raw = full[full['event'] == 'transaction'].copy()
amount_upper = tx_raw['amount'].quantile(0.99)
tx = tx_raw[tx_raw['amount'] <= amount_upper].copy()

print(f"amount 상위 1% 기준값: ${amount_upper:.2f}")
print(f"제거 건수: {len(tx_raw) - len(tx):,}건 ({(len(tx_raw)-len(tx))/len(tx_raw)*100:.2f}%)")

amount 상위 1% 기준값: $40.02
제거 건수: 1,390건 (1.00%)


In [8]:
print("[ 3-1. 소득 구간별 완료율 + 거래금액 ]")

inc_order = ['5만 미만', '5-7.5만', '7.5-10만', '10만 이상']

inc_cr = (
    funnel_clean[funnel_clean['offer_type'].isin(['bogo','discount'])]
    .groupby('income_group')['was_completed'].mean() * 100
).reindex(inc_order)

print("\n소득 구간별 완료율:")
for i, r in inc_cr.items():
    print(f"  {i:<10}  {r:.1f}%")

bd = funnel_clean[funnel_clean['offer_type'].isin(['bogo','discount'])]
ct_inc = pd.crosstab(bd['income_group'], bd['was_completed'])
chi2_inc, p_inc, _, expected_inc = stats.chi2_contingency(ct_inc)
n_inc = ct_inc.values.sum()
v_inc = np.sqrt(chi2_inc / (n_inc * (min(ct_inc.shape)-1)))

print(f"\n기대빈도 최솟값: {expected_inc.min():.1f}  {'(>=5 OK)' if expected_inc.min() >= 5 else '(<5 해석 주의)'}")
print(f"카이제곱: χ²={chi2_inc:.2f},  p={p_inc:.2e},  V={v_inc:.3f}")
print(f"→ {'유의' if p_inc < 0.05 else '비유의'}  효과 크기: {'강함' if v_inc>=0.3 else '중간' if v_inc>=0.1 else '약함'}")

print("\n소득 구간별 거래금액 (transaction 단위):")
inc_tx = tx[tx['income_group'].isin(inc_order)].copy()
inc_amt = inc_tx.groupby('income_group')['amount'].agg(['mean','median']).reindex(inc_order)
for i, row in inc_amt.iterrows():
    print(f"  {i:<10}  평균 ${row['mean']:.2f}  중앙값 ${row['median']:.2f}")

print("\n소득 구간별 1인당 평균 거래금액 (고객 단위):")
person_amt = inc_tx.groupby(['customer_id','income_group'])['amount'].mean().reset_index()
inc_person_amt = person_amt.groupby('income_group')['amount'].mean().reindex(inc_order)
for i, v in inc_person_amt.items():
    print(f"  {i:<10}  ${v:.2f}")

groups_inc = [
    inc_tx[inc_tx['income_group'] == i]['amount'].dropna().values
    for i in inc_order if len(inc_tx[inc_tx['income_group'] == i]) > 0
]

f_inc, p_anova_inc = stats.f_oneway(*groups_inc)
print(f"\nANOVA (거래금액): F={f_inc:.2f},  p={p_anova_inc:.2e}")

kw_inc, p_kw_inc = stats.kruskal(*groups_inc)
print(f"Kruskal-Wallis:  H={kw_inc:.2f},  p={p_kw_inc:.2e}  "
      f"{'(ANOVA 결과와 일치)' if (p_anova_inc < 0.05) == (p_kw_inc < 0.05) else '(ANOVA와 불일치, 비모수 기준 우선)'}")

inc_tx_sub = inc_tx[['income_group','amount']].dropna()
tukey_inc = pairwise_tukeyhsd(inc_tx_sub['amount'], inc_tx_sub['income_group'], alpha=0.05)

print("\nTukey HSD 사후 검정 (유의한 쌍만):")
tukey_df = pd.DataFrame(data=tukey_inc._results_table.data[1:], columns=tukey_inc._results_table.data[0])
sig = tukey_df[tukey_df['reject'] == True][['group1','group2','meandiff','p-adj']]

if len(sig) > 0:
    for _, row in sig.iterrows():
        print(f"  {row['group1']} vs {row['group2']}: 차이 ${float(row['meandiff']):.2f}  p={float(row['p-adj']):.4f}")
else:
    print("  유의한 쌍 없음")

[ 3-1. 소득 구간별 완료율 + 거래금액 ]

소득 구간별 완료율:
  5만 미만       43.8%
  5-7.5만      59.4%
  7.5-10만     74.3%
  10만 이상      76.5%

기대빈도 최솟값: 1574.9  (>=5 OK)
카이제곱: χ²=3006.33,  p=0.00e+00,  V=0.238
→ 유의  효과 크기: 중간

소득 구간별 거래금액 (transaction 단위):
  5만 미만       평균 $5.79  중앙값 $4.42
  5-7.5만      평균 $10.90  중앙값 $10.38
  7.5-10만     평균 $22.21  중앙값 $21.96
  10만 이상      평균 $26.09  중앙값 $26.06

소득 구간별 1인당 평균 거래금액 (고객 단위):
  5만 미만       $5.35
  5-7.5만      $11.16
  7.5-10만     $22.32
  10만 이상      $26.02

ANOVA (거래금액): F=36591.57,  p=0.00e+00
Kruskal-Wallis:  H=52578.23,  p=0.00e+00  (ANOVA 결과와 일치)

Tukey HSD 사후 검정 (유의한 쌍만):
  10만 이상 vs 5-7.5만: 차이 $-15.19  p=0.0000
  10만 이상 vs 5만 미만: 차이 $-20.30  p=0.0000
  10만 이상 vs 7.5-10만: 차이 $-3.88  p=0.0000
  5-7.5만 vs 5만 미만: 차이 $-5.11  p=0.0000
  5-7.5만 vs 7.5-10만: 차이 $11.31  p=0.0000
  5만 미만 vs 7.5-10만: 차이 $16.42  p=0.0000


In [ ]:
print("[ 3-2. 성별 × 오퍼 유형 완료율 ]")

gender_order = ['M', 'F', 'O']
bd_g = funnel_clean[funnel_clean['offer_type'].isin(['bogo','discount']) &
                    funnel_clean['gender'].isin(gender_order)]

heatmap_gender = bd_g.groupby(['gender','offer_type'])['was_completed'].mean().unstack() * 100
heatmap_gender = heatmap_gender.reindex(gender_order)
print("\n완료율 히트맵 (%):")
print(heatmap_gender.round(1))

ct_g = pd.crosstab(bd_g['gender'], bd_g['was_completed'])
chi2_g, p_g, _, expected_g = stats.chi2_contingency(ct_g)
v_g = np.sqrt(chi2_g / (ct_g.values.sum() * (min(ct_g.shape)-1)))
print(f"\n기대빈도 최솟값: {expected_g.min():.1f}  {'(>=5 OK)' if expected_g.min() >= 5 else '(<5 해석 주의)'}")
print(f"χ²={chi2_g:.2f},  p={p_g:.2e},  V={v_g:.3f}  → {'유의' if p_g<0.05 else '비유의'}")

print("\n오퍼 유형별 성별 완료율 + 95% CI:")
for ot in ['bogo', 'discount']:
    sub = bd_g[bd_g['offer_type'] == ot]
    for g in ['F', 'M']:
        g_sub = sub[sub['gender'] == g]
        n = len(g_sub)
        k = g_sub['was_completed'].sum()
        rate = k / n * 100
        lo, hi = proportion_confint(k, n, alpha=0.05, method='wilson')
        print(f"  {ot:<12} {g}  {rate:.1f}%  95%CI [{lo*100:.1f}%, {hi*100:.1f}%]  (n={n:,})")

print("\n오퍼 유형별 성별 완료율 차이:")
for ot in ['bogo','discount']:
    sub = bd_g[bd_g['offer_type']==ot]
    ct_sub = pd.crosstab(sub['gender'], sub['was_completed'])
    chi2_s, p_s, _, _ = stats.chi2_contingency(ct_sub)
    f_cr = sub[sub['gender']=='F']['was_completed'].mean()*100
    m_cr = sub[sub['gender']=='M']['was_completed'].mean()*100
    print(f"  {ot:<12}  F:{f_cr:.1f}%  M:{m_cr:.1f}%  차이:{f_cr-m_cr:.1f}%p  p={p_s:.2e}")

In [ ]:
print("[ 3-3. 연령대 × 오퍼 유형 완료율 ]")

age_order = ['20대 미만', '20대', '30대', '40대', '50대', '60대 이상']

bd_a = funnel_clean[
    (funnel_clean['offer_type'].isin(['bogo','discount'])) &
    (funnel_clean['age_group'].isin(age_order))
].copy()

heatmap_age = (
    bd_a.groupby(['age_group','offer_type'])['was_completed']
    .mean().unstack() * 100
).reindex(age_order)

print("\n완료율 히트맵 (%):")
print(heatmap_age.round(1))

ct_a = pd.crosstab(bd_a['age_group'], bd_a['was_completed'])
chi2_a, p_a, _, expected_a = stats.chi2_contingency(ct_a)
v_a = np.sqrt(chi2_a / (ct_a.values.sum() * (min(ct_a.shape)-1)))

print(f"\n기대빈도 최솟값: {expected_a.min():.1f}  {'(>=5 OK)' if expected_a.min() >= 5 else '(<5 해석 주의)'}")
print(f"카이제곱: χ²={chi2_a:.2f},  p={p_a:.2e},  V={v_a:.3f}  → {'유의' if p_a<0.05 else '비유의'}")
print(f"효과 크기: {'강함' if v_a>=0.3 else '중간' if v_a>=0.1 else '약함'}")

print("\n연령대 × 오퍼 유형 완료율 + 95% CI:")
for ot in ['bogo', 'discount']:
    print(f"  [{ot}]")
    for age in age_order:
        sub = bd_a[(bd_a['offer_type'] == ot) & (bd_a['age_group'] == age)]
        n = len(sub)
        k = sub['was_completed'].sum()
        rate = k / n * 100
        lo, hi = proportion_confint(k, n, alpha=0.05, method='wilson')
        print(f"    {age:<8}  {rate:.1f}%  95%CI [{lo*100:.1f}%, {hi*100:.1f}%]  (n={n:,})")

print("\n연령대별 discount - bogo 완료율 차이:")
diff = heatmap_age.get('discount', 0) - heatmap_age.get('bogo', 0)
for age, d in diff.items():
    print(f"  {age:<8}  {d:+.1f}%p")

In [ ]:
print("[ 3-3-1. 나이_성별 × 오퍼 유형 완료율 ]")

age_gender_order = [
    '20세 미만 남성', '20세 미만 여성',
    '20대 남성', '20대 여성',
    '30대 남성', '30대 여성',
    '40대 남성', '40대 여성',
    '50대 남성', '50대 여성',
    '60대 남성', '60대 여성',
    '60+ 남성',  '60+ 여성',
]

bd_ag = funnel_clean[
    (funnel_clean['offer_type'].isin(['bogo', 'discount'])) &
    (funnel_clean['age_gender'].isin(age_gender_order))
].copy()

heatmap_ag = (
    bd_ag.groupby(['age_gender', 'offer_type'])['was_completed']
    .mean().unstack() * 100
).reindex(age_gender_order)

n_ag = (
    bd_ag.groupby(['age_gender', 'offer_type'])['was_completed']
    .count().unstack()
).reindex(age_gender_order)

print("\n완료율 히트맵 (%) — 괄호 안은 표본수 n:")
for ag in age_gender_order:
    row_parts = []
    for ot in ['bogo', 'discount']:
        rate = heatmap_ag.loc[ag, ot] if (ag in heatmap_ag.index and ot in heatmap_ag.columns) else float('nan')
        n = n_ag.loc[ag, ot] if (ag in n_ag.index and ot in n_ag.columns) else 0
        if pd.notna(rate):
            flag = " ⚠" if n < 100 else ""
            row_parts.append(f"{ot}: {rate:.1f}% (n={int(n)}){flag}")
        else:
            row_parts.append(f"{ot}: -")
    print(f"  {ag:<12}  " + "  ".join(row_parts))

print("\n※ ⚠ 표본수 100 미만 — 비율 해석 주의")

ct_ag = pd.crosstab(bd_ag['age_gender'], bd_ag['was_completed'])
chi2_ag, p_ag, _, expected_ag = stats.chi2_contingency(ct_ag)
v_ag = np.sqrt(chi2_ag / (ct_ag.values.sum() * (min(ct_ag.shape) - 1)))

print(f"\n기대빈도 최솟값: {expected_ag.min():.1f}  {'(>=5 OK)' if expected_ag.min() >= 5 else '(<5 해석 주의)'}")
print(f"카이제곱: χ²={chi2_ag:.2f},  p={p_ag:.2e},  V={v_ag:.3f}  → {'유의' if p_ag < 0.05 else '비유의'}")
print(f"효과 크기: {'강함' if v_ag >= 0.3 else '중간' if v_ag >= 0.1 else '약함'}")

print("\n나이_성별별 discount - bogo 완료율 차이:")
diff_ag = heatmap_ag.get('discount', 0) - heatmap_ag.get('bogo', 0)
for ag, d in diff_ag.items():
    print(f"  {ag:<12}  {d:+.1f}%p")

In [15]:
if 'join_year' not in funnel_clean.columns:
    join_attr = full[['customer_id', 'join_year']].drop_duplicates('customer_id')
    funnel_clean = funnel_clean.merge(join_attr, on='customer_id', how='left')

if 'join_year' not in tx.columns:
    join_attr = full[['customer_id', 'join_year']].drop_duplicates('customer_id')
    tx = tx.merge(join_attr, on='customer_id', how='left')

print("[ 3-4. 가입년도별 오퍼 완료율 ]")

cohort_order = sorted(funnel_clean['join_year'].dropna().unique())
fd = funnel_clean[funnel_clean['offer_type'].isin(['bogo', 'discount'])].copy()

cohort_cr = fd.groupby('join_year')['was_completed'].mean() * 100
cohort_cr = cohort_cr.reindex(cohort_order)

print("\n가입년도별 완료율:")
for y, r in cohort_cr.items():
    print(f"  {int(y)}년  {r:.1f}%")

ct_c = pd.crosstab(fd['join_year'], fd['was_completed'])
chi2_c, p_c, _, _ = stats.chi2_contingency(ct_c)
v_c = np.sqrt(chi2_c / (ct_c.values.sum() * (min(ct_c.shape) - 1)))

print(f"카이제곱: χ²={chi2_c:.2f},  p={p_c:.2e},  V={v_c:.3f}")
print(f"→ {'유의' if p_c < 0.05 else '비유의'}  효과 크기: {'강함' if v_c>=0.3 else '중간' if v_c>=0.1 else '약함'}")

print("가입년도별 평균 거래금액:")
cohort_tx = tx[tx['join_year'].isin(cohort_order)].copy()
cohort_amt = cohort_tx.groupby('join_year')['amount'].agg(['mean', 'median']).reindex(cohort_order)
for y, row in cohort_amt.iterrows():
    print(f"  {int(y)}년  평균 ${row['mean']:.2f}  중앙값 ${row['median']:.2f}")

[ 3-4. 가입년도별 오퍼 완료율 ]

가입년도별 완료율:
  2013년  56.3%
  2014년  54.8%
  2015년  74.0%
  2016년  81.6%
  2017년  59.7%
  2018년  38.5%
카이제곱: χ²=5160.48,  p=0.00e+00,  V=0.311
→ 유의  효과 크기: 강함
가입년도별 평균 거래금액:
  2013년  평균 $5.75  중앙값 $3.17
  2014년  평균 $5.80  중앙값 $3.14
  2015년  평균 $11.10  중앙값 $8.77
  2016년  평균 $13.07  중앙값 $12.25
  2017년  평균 $11.67  중앙값 $9.89
  2018년  평균 $9.26  중앙값 $5.97


In [13]:
# 성별 × 오퍼 유형 히트맵용 데이터 먼저 생성
bd_g = funnel_clean[
    (funnel_clean['offer_type'].isin(['bogo', 'discount'])) &
    (funnel_clean['gender'].notna())
].copy()

heatmap_gender = (
    bd_g.groupby(['gender', 'offer_type'])['was_completed']
    .mean()
    .unstack() * 100
)

# 카이제곱
ct_g = pd.crosstab(bd_g['gender'], bd_g['was_completed'])
chi2_g, p_g, _, _ = stats.chi2_contingency(ct_g)
v_g = np.sqrt(chi2_g / (ct_g.values.sum() * (min(ct_g.shape) - 1)))

In [ ]:
# ----------
# 시각화 (3막)
# ----------
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Act 3 — Segment Analysis', fontsize=14, fontweight='bold')

# 차트 1: 소득 구간별 완료율
ax1 = axes[0, 0]
cr_vals = [inc_cr.get(i, 0) for i in inc_order]
bars1 = ax1.bar(inc_order, cr_vals, color=['#D4E9E2', '#CBA258', '#00704A', '#1E3932'], alpha=0.9)
for bar, v in zip(bars1, cr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{v:.1f}%', ha='center', fontsize=10)
ax1.set_title('Completion Rate by Income Group', fontweight='bold')
ax1.set_ylabel('Completion Rate (%)')
ax1.set_ylim(0, max(cr_vals) * 1.2 if max(cr_vals) > 0 else 1)

# 차트 2: 소득 구간별 평균 거래금액
ax2 = axes[0, 1]
inc_mean = [inc_amt.loc[i, 'mean'] if i in inc_amt.index else 0 for i in inc_order]
bars2 = ax2.bar(inc_order, inc_mean, color=['#D4E9E2', '#CBA258', '#00704A', '#1E3932'], alpha=0.9)
for bar, v in zip(bars2, inc_mean):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'${v:.1f}', ha='center', fontsize=9)
ax2.set_title('Avg Transaction Amount by Income', fontweight='bold')
ax2.set_ylabel('Amount ($)')
ax2.set_ylim(0, max(inc_mean) * 1.2 if max(inc_mean) > 0 else 1)

# 차트 3: 성별 × 오퍼 유형 히트맵 (rate + n)
ax3 = axes[0, 2]
gender_order_vis = ['M', 'F', 'O']
hm_g = heatmap_gender.reindex(index=gender_order_vis, columns=['bogo', 'discount'])
n_g_hm = (
    bd_g.groupby(['gender', 'offer_type'])['was_completed']
    .count().unstack()
    .reindex(index=gender_order_vis, columns=['bogo', 'discount'])
)
annot_g = pd.DataFrame(
    [[f"{hm_g.loc[r, c]:.1f}%\nn={int(n_g_hm.loc[r, c])}"
      if (pd.notna(hm_g.loc[r, c]) and pd.notna(n_g_hm.loc[r, c])) else ''
      for c in hm_g.columns] for r in hm_g.index],
    index=hm_g.index, columns=hm_g.columns
)
sns.heatmap(
    hm_g, annot=annot_g, fmt='', cmap='YlGn',
    ax=ax3, linewidths=0.5, cbar_kws={'label': 'Completion Rate (%)'}
)
ax3.set_title(f'Completion Rate: Gender × Offer Type\np={p_g:.2e}', fontweight='bold')
ax3.set_xlabel('Offer Type')
ax3.set_ylabel('Gender')

# 차트 4: 연령대 × 오퍼 유형 히트맵 (rate + n)
ax4 = axes[1, 0]
hm_a = heatmap_age.reindex(columns=['bogo', 'discount'])
n_a_hm = (
    bd_a.groupby(['age_group', 'offer_type'])['was_completed']
    .count().unstack()
    .reindex(index=age_order, columns=['bogo', 'discount'])
)
annot_a = pd.DataFrame(
    [[f"{hm_a.loc[r, c]:.1f}%\nn={int(n_a_hm.loc[r, c])}"
      if (pd.notna(hm_a.loc[r, c]) and pd.notna(n_a_hm.loc[r, c])) else ''
      for c in hm_a.columns] for r in hm_a.index],
    index=hm_a.index, columns=hm_a.columns
)
sns.heatmap(
    hm_a, annot=annot_a, fmt='', cmap='YlGn',
    ax=ax4, linewidths=0.5, cbar_kws={'label': 'Completion Rate (%)'}
)
ax4.set_title(f'Completion Rate: Age × Offer Type\np={p_a:.2e}', fontweight='bold')
ax4.set_xlabel('Offer Type')
ax4.set_ylabel('Age Group')

# 차트 5: 가입년도별 평균 거래금액
ax5 = axes[1, 1]
cohort_labels = [str(int(y)) for y in cohort_order]
cohort_mean = [cohort_amt.loc[y, 'mean'] if y in cohort_amt.index else 0 for y in cohort_order]
bars5 = ax5.bar(cohort_labels, cohort_mean, color=['#D4E9E2', '#CBA258', '#00704A', '#1E3932'][:len(cohort_labels)], alpha=0.9)
for bar, v in zip(bars5, cohort_mean):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'${v:.1f}', ha='center', fontsize=9)
ax5.set_title('Avg Transaction Amount by Join Year', fontweight='bold')
ax5.set_ylabel('Amount ($)')
ax5.set_ylim(0, max(cohort_mean) * 1.2 if max(cohort_mean) > 0 else 1)

# 차트 6: 가입년도별 완료율
ax6 = axes[1, 2]
cohort_cr_vals = [cohort_cr.get(y, 0) for y in cohort_order]
bars6 = ax6.bar(cohort_labels, cohort_cr_vals, color=['#D4E9E2', '#CBA258', '#00704A', '#1E3932'][:len(cohort_labels)], alpha=0.9)
for bar, v in zip(bars6, cohort_cr_vals):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{v:.1f}%', ha='center', fontsize=10)
ax6.set_title(f'Completion Rate by Join Year\np={p_c:.2e}', fontweight='bold')
ax6.set_ylabel('Completion Rate (%)')
ax6.set_ylim(0, max(cohort_cr_vals) * 1.2 if max(cohort_cr_vals) > 0 else 1)

plt.tight_layout()
plt.show()